# Modified pix2pixHD — Training

This notebook trains pix2pixHD with geographic XYZ metadata supplied separately from RGB images and introduced at the generator bottleneck. Only `bottleneck` conditioning is supported; both `global` and `local` generators are available.

Coordinate metadata is required. Set `COORD_CSV` to a CSV path or place `geo_coordinates.csv` inside `DATAROOT`. The file can be prepared using `generate_coordinate_csv.ipynb`.

Dataset paths, experiment name, GPU selection, and training parameters are configured in the cells below. Use a distinct experiment name for each training configuration.

Training parameters such as batch size, learning rate, number of epochs, normalization, generator/discriminator configuration, and checkpoint intervals are defined in the settings cells below.

In [ ]:
from pathlib import Path
import sys
import subprocess

# Open the notebook from the project root or the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROJECT_ROOT to the project directory."
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)


In [ ]:
# Edit only the settings in this cell.

DATAROOT = PROJECT_ROOT / "datasets"
NAME = "Model_V2_dataset_global_D3_allloss_aug_batch_coord_bottleneck"
# Use a distinct experiment name for each training configuration.
COORD_CSV = ""  # Empty: DATAROOT/geo_coordinates.csv
GPU_IDS = "0"  # Use "-1" for CPU (a GPU is recommended for training)
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"


In [ ]:
settings = {
    "name": NAME,
    "dataroot": str(DATAROOT),
    "checkpoints_dir": str(CHECKPOINTS_DIR),
    "gpu_ids": GPU_IDS,

    "label_nc": 0,
    "input_nc": 3,
    "output_nc": 3,

    "loadSize": 256,
    "fineSize": 256,
    "resize_or_crop": "none",

    "netG": "global",
    "n_blocks_global": 9,
    "ngf": 64,
    "n_downsample_global": 4,
    "norm": "batch",
}

if COORD_CSV:
    settings["coord_csv"] = COORD_CSV

flags = ["no_instance"]

settings.update({
    "batchSize": 32,
    "num_D": 3,
    "n_layers_D": 3,
    "ndf": 64,
    "lambda_feat": 10.0,
    "niter": 100,
    "niter_decay": 100,
    "lr": 0.0002,
    "beta1": 0.5,
    "nThreads": 8,
    "save_epoch_freq": 20,
    "save_latest_freq": 282400,
    "display_freq": 564800,
    "print_freq": 160,
})


In [ ]:
assert DATAROOT.is_dir(), f"Dataset directory not found: {DATAROOT}"
csv_path = Path(COORD_CSV) if COORD_CSV else DATAROOT / "geo_coordinates.csv"
assert csv_path.is_file(), f"Coordinate CSV not found: {csv_path}"
assert (DATAROOT / "train_A").is_dir() and (DATAROOT / "train_B").is_dir()
if "continue_train" not in flags:
    assert not list((CHECKPOINTS_DIR / NAME).glob("*_net_*.pth")), "This experiment already contains weights; choose a new NAME or intentionally enable continue_train."
command = [sys.executable, "-u", str(PROJECT_ROOT / "train.py")]
for key, value in settings.items():
    command.extend(["--" + key, str(value)])
command.extend("--" + flag for flag in flags)
print(subprocess.list2cmdline(command))


In [ ]:
# This cell starts the process. Interrupt the kernel to stop it.
process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
    if returncode:
        raise RuntimeError(f"Process exited with error code: {returncode}")
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
